# 特征工程

我们先捋一下基于原始的给定数据， 有哪些特征可以直接利用：

1. 文章的自身特征， category_id表示这文章的类型， created_at_ts表示文章建立的时间， 这个关系着文章的时效性， words_count是文章的字数， 一般字数太长我们不太喜欢点击, 也不排除有人就喜欢读长文。
2. 文章的内容embedding特征， 这个召回的时候用过， 这里可以选择使用， 也可以选择不用， 也可以尝试其他类型的embedding特征， 比如W2V等
3. 用户的设备特征信息

上面这些直接可以用的特征， 待做完特征工程之后， 直接就可以根据article_id或者是user_id把这些特征加入进去。 但是我们需要先基于召回的结果， 构造一些特征，然后制作标签，形成一个监督学习的数据集。<br><br>
构造监督数据集的思路， 根据召回结果， 我们会得到一个{user_id: [可能点击的文章列表]}形式的字典。 那么我们就可以对于每个用户， 每篇可能点击的文章构造一个监督测试集， 比如对于用户user1， 假设得到的他的召回列表{user1: [item1, item2, item3]}， 我们就可以得到三行数据(user1, item1), (user1, item2), (user1, item3)的形式， 这就是监督测试集时候的前两列特征。<br><br>

构造特征的思路是这样， 我们知道每个用户的点击文章是与其历史点击的文章信息是有很大关联的， 比如同一个主题， 相似等等。 所以特征构造这块很重要的一系列特征**是要结合用户的历史点击文章信息**。我们已经得到了每个用户及点击候选文章的两列的一个数据集， 而我们的目的是要预测最后一次点击的文章， 比较自然的一个思路就是和其最后几次点击的文章产生关系， 这样既考虑了其历史点击文章信息， 又得离最后一次点击较近，因为新闻很大的一个特点就是注重时效性。 往往用户的最后一次点击会和其最后几次点击有很大的关联。 所以我们就可以对于每个候选文章， 做出与最后几次点击相关的特征如下：

1. 候选item与最后几次点击的相似性特征(embedding内积）  --- 这个直接关联用户历史行为
2. 候选item与最后几次点击的相似性特征的统计特征 --- 统计特征可以减少一些波动和异常
3. 候选item与最后几次点击文章的字数差的特征 --- 可以通过字数看用户偏好
4. 候选item与最后几次点击的文章建立的时间差特征 --- 时间差特征可以看出该用户对于文章的实时性的偏好   

还需要考虑一下
**5. 如果使用了youtube召回的话， 我们还可以制作用户与候选item的相似特征**

当然， 上面只是提供了一种基于用户历史行为做特征工程的思路， 大家也可以思维风暴一下，尝试一些其他的特征。 下面我们就实现上面的这些特征的制作， 下面的逻辑是这样：

1. 我们首先获得用户的最后一次点击操作和用户的历史点击， 这个基于我们的日志数据集做
2. 基于用户的历史行为制作特征， 这个会用到用户的历史点击表， 最后的召回列表， 文章的信息表和embedding向量
3. 制作标签， 形成最后的监督学习数据集

## 导包

In [58]:
import gc, os
import logging
import pickle
import time
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
import numpy as np
import pandas as pd
from tqdm import tqdm
import lightgbm as lgb
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# 🎯 核心配置：选择运行模式（必须与 Recall 阶段保持一致！）
# ============================================================================
# "offline" = 离线验证模式：用于评估排序效果，使用划分好的训练/验证集
#             ⚠️ 数据泄漏防护：特征只使用"历史"数据，不使用验证集标签
# "online"  = 在线提交模式：用于生成最终排序结果，使用全量数据
# ============================================================================
MODE = "offline"  # 👈 修改这里切换模式（需要与 4.recall.ipynb 保持一致！）

print("=" * 70)
print(f"🔧 特征工程 - 当前运行模式: {MODE}")
if MODE == "offline":
    print("🔬 离线验证模式: 会严格区分训练/验证用户，防止数据泄漏")
    print("   ⚠️ 统计特征只使用历史点击（排除标签），确保无未来信息泄漏")
else:
    print("🚀 在线提交模式: 使用全量数据构造特征")
print("=" * 70)

🔧 特征工程 - 当前运行模式: offline
🔬 离线验证模式: 会严格区分训练/验证用户，防止数据泄漏
   ⚠️ 统计特征只使用历史点击（排除标签），确保无未来信息泄漏


## df节省内存函数

In [59]:
# 节省内存的一个函数
# 减少内存
def reduce_mem(df):
    starttime = time.time()
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.isnull(c_min) or pd.isnull(c_max):
                continue
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print('-- Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction),time spend:{:2.2f} min'.format(end_mem,
                                                                                                           100*(start_mem-end_mem)/start_mem,
                                                                                                           (time.time()-starttime)/60))
    return df


## 定义数据路径

In [60]:
# 数据路径 - 使用环境变量
from funrec.utils import load_env_with_fallback
load_env_with_fallback()

# 原始数据路径
RAW_DATA_PATH = Path(os.getenv('FUNREC_RAW_DATA_PATH'))
data_path = RAW_DATA_PATH / 'news_recommendation'

# 处理后数据路径（中间结果、模型文件等）
PROCESSED_DATA_PATH = Path(os.getenv('FUNREC_PROCESSED_DATA_PATH'))
base_path = PROCESSED_DATA_PATH / 'projects/news_recommendation'
save_path = base_path / 'temp_results'
if not save_path.exists():
    save_path.mkdir(parents=True, exist_ok=True)

print(f"📁 原始数据路径: {data_path}")
print(f"📁 结果保存路径: {save_path}")

📁 原始数据路径: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/data/news_recommendation
📁 结果保存路径: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results


## 数据读取

### 训练和验证集的划分

划分训练和验证集的原因是为了在线下验证模型参数的好坏，为了完全模拟测试集，我们这里就在训练集中抽取部分用户的所有信息来作为验证集。提前做训练验证集划分的好处就是可以分解制作排序特征时的压力，一次性做整个数据集的排序特征可能时间会比较长。

## ⚠️ 数据泄漏防护设计

### 数据流架构（离线验证模式）

```
原始训练集 (train_click_log.csv)
    │
    ├── 训练用户 (click_trn)
    │       ├── 历史点击 (click_trn_hist)  ─→ 用于构造特征 ✅
    │       └── 最后一次点击 (click_trn_last)  ─→ 仅用于打标签 ⚠️
    │
    └── 验证用户 (click_val + val_ans)
            ├── 历史点击 (click_val_hist = click_val)  ─→ 用于构造特征 ✅
            └── 最后一次点击 (click_val_last = val_ans)  ─→ 仅用于评估 ⚠️
```

### 关键防护点

| 环节 | 使用的数据 | ⚠️ 禁止使用 |
|------|-----------|------------|
| 计算用户活跃度特征 | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 计算文章热度特征 | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 训练 Word2Vec | click_trn_hist + click_val_hist | click_trn_last, val_ans |
| 构造交互特征 | 用户各自的历史点击 | 用户各自的最后一次点击 |
| 打标签 | click_trn_last, val_ans | - |

### 与 Recall 阶段的一致性

⚠️ **重要**：特征工程必须与 Recall 阶段使用相同的数据划分！

- 直接读取 Recall 保存的 `click_trn.csv`, `click_val.csv`, `val_ans.csv`
- 确保训练用户/验证用户的划分完全一致
- MODE 设置必须与 Recall 阶段保持一致

In [61]:
# ============================================================================
# 📊 数据读取函数
# ============================================================================
# ⚠️ 关键设计：
#   - 离线模式：直接读取 Recall 阶段保存的 click_trn, click_val, val_ans
#   - 在线模式：训练集 + 测试集全部数据
# ============================================================================

def get_all_click_df(data_path, mode="offline"):
    """
    读取原始点击数据
    
    参数:
        data_path: 数据路径
        mode: 运行模式
            - "offline": 离线验证模式，只使用训练集
            - "online": 在线提交模式，合并训练集和测试集
    
    返回:
        all_click: 点击数据DataFrame
    """
    if mode == "offline":
        all_click = pd.read_csv(data_path / 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path / 'train_click_log.csv')
        tst_click = pd.read_csv(data_path / 'testA_click_log.csv')
        all_click = pd.concat([trn_click, tst_click]).reset_index(drop=True)
    
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

In [62]:
# 采样数据（仅用于原始数据参考，实际使用 Recall 保存的划分）
all_click_df = get_all_click_df(data_path, mode=MODE)

这部分是我加的

In [63]:
# ============================================================================
# 🔬 离线模式：训练集和验证集划分（直接复用 Recall 阶段的划分！）
# ============================================================================
# ⚠️ 重要说明：
#   为了保证与 Recall 阶段完全一致，特征工程直接读取 Recall 保存的文件，
#   而不是重新划分。这样可以确保：
#   1. 训练用户 vs 验证用户的划分完全一致
#   2. 验证用户的历史点击 vs 最后一次点击（标签）划分一致
# ============================================================================

def trn_val_split(all_click_df, sample_user_nums):
    """
    将训练集划分为：训练用户 + 验证用户
    （⚠️ 仅在需要重新划分时使用，一般直接读取 Recall 阶段保存的文件）
    """
    all_click = all_click_df
    all_user_ids = all_click.user_id.unique()
    
    sample_user_ids = np.random.choice(all_user_ids, size=sample_user_nums, replace=False) 
    
    click_val = all_click[all_click['user_id'].isin(sample_user_ids)]
    click_trn = all_click[~all_click['user_id'].isin(sample_user_ids)]
    
    # 将验证用户的最后一次点击提取出来作为 ground truth
    click_val = click_val.sort_values(['user_id', 'click_timestamp'])
    val_ans = click_val.groupby('user_id').tail(1)
    
    # 验证用户去掉最后一次点击，剩下的作为历史
    click_val = click_val.groupby('user_id').apply(lambda x: x[:-1]).reset_index(drop=True)
    
    # 去除只有一条点击的用户
    val_ans = val_ans[val_ans.user_id.isin(click_val.user_id.unique())]
    click_val = click_val[click_val.user_id.isin(val_ans.user_id.unique())]
    
    return click_trn, click_val, val_ans

### 获取历史点击和最后一次点击，这里和前面对单条数据的处理不太一样，前面是直接删掉单条数据，这里是将单条数据作为指标，也作为最后一次点击。

In [64]:
# ============================================================================
# 📊 获取历史点击和最后一次点击
# ============================================================================
# ⚠️ 数据泄漏防护：
#   - 历史点击（click_hist_df）：用于构造特征
#   - 最后一次点击（click_last_df）：用于打标签（Label），不能用于构造特征！
# ============================================================================

def get_hist_and_last_click(all_click):
    """
    获取用户的历史点击和最后一次点击
    
    返回:
        click_hist_df: 历史点击（用于构造特征）
        click_last_df: 最后一次点击（用于打标签，⚠️不能泄漏到特征中！）
    """
    all_click = all_click.copy()
    
    # ⚠️ 修复 float16 不支持排序的问题：将 click_timestamp 转为 float32
    if all_click['click_timestamp'].dtype == np.float16:
        all_click['click_timestamp'] = all_click['click_timestamp'].astype(np.float32)
    
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # 如果用户只有一个点击，hist为空会导致训练时用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df

### 读取训练、验证及测试集（原始的）,返回的是按用户分组后的训练集和验证集

In [65]:
# ============================================================================
# 📊 读取训练、验证及测试集（直接复用 Recall 阶段的划分！）
# ============================================================================
# ⚠️ 重要：直接读取 Recall 保存的文件，确保划分一致！
# ============================================================================

def get_trn_val_tst_data(data_path, save_path, mode="offline"):
    """
    读取训练、验证及测试集
    
    参数:
        data_path: 原始数据路径
        save_path: Recall 阶段保存的中间结果路径
        mode: 运行模式 ("offline" / "online")
    
    返回:
        click_trn: 训练用户的点击数据
        click_val: 验证用户的历史点击（不含最后一次）
        click_tst: 测试集用户的点击数据
        val_ans: 验证用户的最后一次点击（标签）
    """
    if mode == "offline":
        # ═══════════════════════════════════════════════════════════════════
        # 🔬 离线验证模式：直接读取 Recall 阶段保存的切分文件
        # ═══════════════════════════════════════════════════════════════════
        print("📂 [离线模式] 读取 Recall 阶段保存的数据划分...")
        click_trn = pd.read_csv(save_path / 'click_trn.csv')
        click_val = pd.read_csv(save_path / 'click_val.csv')
        val_ans = pd.read_csv(save_path / 'val_ans.csv')
        click_trn = reduce_mem(click_trn)
        click_val = reduce_mem(click_val)
        val_ans = reduce_mem(val_ans)
        print(f"   ✅ 训练用户数: {click_trn['user_id'].nunique()}")
        print(f"   ✅ 验证用户数: {click_val['user_id'].nunique()}")
        print(f"   ✅ 验证集答案数: {len(val_ans)}")
    else:
        # ═══════════════════════════════════════════════════════════════════
        # 🚀 在线提交模式：使用全量训练数据
        # ═══════════════════════════════════════════════════════════════════
        print("📂 [在线模式] 读取全量训练数据...")
        click_trn = pd.read_csv(data_path / 'train_click_log.csv')
        click_trn = reduce_mem(click_trn)
        click_val = None
        val_ans = None
        print(f"   ✅ 训练数据量: {len(click_trn)}")
    
    click_tst = pd.read_csv(data_path / 'testA_click_log.csv')
    print(f"   ✅ 测试集用户数: {click_tst['user_id'].nunique()}")
    
    return click_trn, click_val, click_tst, val_ans

### 读取召回列表

In [66]:
# ============================================================================
# 📊 读取召回列表（直接复用 Recall 阶段的结果）
# ============================================================================
# ⚠️ 注意：Recall 阶段已经按照离线/在线模式分别保存了召回列表
#          这里直接读取即可
# ============================================================================

def get_recall_list(save_path, single_recall_model=None, multi_recall=True):
    """
    读取召回列表
    
    返回格式: {user_id: [(item_id, score), ...]}
    """
    if multi_recall:
        recall_path = save_path / 'final_recall_items_dict.pkl'
        print(f"📂 读取多路召回列表: {recall_path}")
        return pickle.load(open(recall_path, 'rb'))
    
    if single_recall_model == 'i2i_itemcf':
        return pickle.load(open(save_path / 'itemcf_recall_dict.pkl', 'rb'))
    elif single_recall_model == 'i2i_emb_itemcf':
        return pickle.load(open(save_path / 'itemcf_emb_dict.pkl', 'rb'))
    elif single_recall_model == 'user_cf':
        return pickle.load(open(save_path / 'youtubednn_usercf_dict.pkl', 'rb'))
    elif single_recall_model == 'youtubednn':
        return pickle.load(open(save_path / 'youtube_u2i_dict.pkl', 'rb'))

### 读取各种Embedding

#### Word2Vec训练及gensim的使用

Word2Vec主要思想是：一个词的上下文可以很好的表达出词的语义。通过无监督学习产生词向量的方式。word2vec中有两个非常经典的模型：skip-gram和cbow。

- skip-gram：已知中心词预测周围词。
- cbow：已知周围词预测中心词。

在使用gensim训练word2vec的时候，有几个比较重要的参数
- size: 表示词向量的维度。
- window：决定了目标词会与多远距离的上下文产生关系。
- sg: 如果是0，则是CBOW模型，是1则是Skip-Gram模型。
- workers: 表示训练时候的线程数量
- min_count: 设置最小的
- iter: 训练时遍历整个数据集的次数

**注意**
1. 训练的时候输入的语料库一定要是字符组成的二维数组，如：[['北', '京', '你', '好'], ['上', '海', '你', '好']]
2. 使用模型的时候有一些默认值，可以通过在Jupyter里面通过`Word2Vec??`查看


下面是个简单的测试样例：

In [67]:
from gensim.models import Word2Vec
docs = [['30760', '157507'],
       ['289197', '63746'],
       ['36162', '168401'],
       ['50644', '36162']]
w2v = Word2Vec(docs, vector_size=12, sg=1, window=2, seed=2020, workers=2, min_count=1, epochs=1)

# 查看'30760'表示的词向量
w2v.wv['30760']


array([ 0.02408178, -0.00102527,  0.08320773, -0.05805062, -0.00330455,
        0.07352284,  0.00942789, -0.05198064, -0.00500923, -0.08016557,
       -0.08009075,  0.07415534], dtype=float32)

skip-gram和cbow的详细原理可以参考下面的博客：
- [word2vec原理(一) CBOW与Skip-Gram模型基础](https://www.cnblogs.com/pinard/p/7160330.html)   
- [word2vec原理(二) 基于Hierarchical Softmax的模型](https://www.cnblogs.com/pinard/p/7160330.html)   
- [word2vec原理(三) 基于Negative Sampling的模型](https://www.cnblogs.com/pinard/p/7249903.html)

In [68]:
def trian_item_word2vec(click_df, embed_size=64, save_name='item_w2v_emb.pkl', split_char=' '):
    click_df = click_df.sort_values('click_timestamp')

    # 只有转换成字符串才可以进行训练
    click_df['click_article_id'] = click_df['click_article_id'].astype(str)
    
    # 转换成句子的形式
    docs = click_df.groupby(['user_id'])['click_article_id'].apply(lambda x: list(x)).reset_index()
    docs = docs['click_article_id'].values.tolist()

    # 为了方便查看训练的进度，这里设定一个log信息
    logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s', level=logging.INFO)

    # 这里的参数对训练得到的向量影响也很大,默认负采样为5
    model = Word2Vec(docs, vector_size=16, sg=1, window=5, seed=2020, workers=24, min_count=1, epochs=1)
    
    # 保存成字典的形式
    item_w2v_emb_dict = {k: model.wv[k] for k in click_df['click_article_id']}
    pickle.dump(item_w2v_emb_dict, open(save_path / 'item_w2v_emb.pkl', 'wb'))
    
    return item_w2v_emb_dict


In [69]:
# 可以通过字典查询对应的item的Embedding
def get_embedding(save_path, all_click_df):
    if os.path.exists(save_path / 'item_content_emb.pkl'):
        item_content_emb_dict = pickle.load(open(save_path / 'item_content_emb.pkl', 'rb'))
    else:
        print('item_content_emb.pkl 文件不存在...')
        
    # w2v Embedding是需要提前训练好的
    if os.path.exists(save_path / 'item_w2v_emb.pkl'):
        item_w2v_emb_dict = pickle.load(open(save_path / 'item_w2v_emb.pkl', 'rb'))
    else:
        item_w2v_emb_dict = trian_item_word2vec(all_click_df)
        
    if os.path.exists(save_path / 'item_youtube_emb.pkl'):
        item_youtube_emb_dict = pickle.load(open(save_path / 'item_youtube_emb.pkl', 'rb'))
    else:
        print('item_youtube_emb.pkl 文件不存在...')
    
    if os.path.exists(save_path / 'user_youtube_emb.pkl'):
        user_youtube_emb_dict = pickle.load(open(save_path / 'user_youtube_emb.pkl', 'rb'))
    else:
        print('user_youtube_emb.pkl 文件不存在...')
    
    return item_content_emb_dict, item_w2v_emb_dict, item_youtube_emb_dict, user_youtube_emb_dict


### 读取文章信息

In [70]:
def get_article_info_df():
    article_info_df = pd.read_csv(data_path / 'articles.csv')
    article_info_df = reduce_mem(article_info_df)
    
    return article_info_df


### 读取数据

In [71]:
# ============================================================================
# 📊 读取数据（根据 MODE 自动选择）
# ============================================================================
click_trn, click_val, click_tst, val_ans = get_trn_val_tst_data(data_path, save_path, mode=MODE)

print("\n" + "=" * 70)
print("📊 数据加载完成")
print("=" * 70)

📂 [离线模式] 读取 Recall 阶段保存的数据划分...
-- Mem. usage decreased to 13.57 Mb (77.8% reduction),time spend:0.00 min
-- Mem. usage decreased to  2.80 Mb (77.8% reduction),time spend:0.00 min
-- Mem. usage decreased to  0.84 Mb (69.4% reduction),time spend:0.00 min
   ✅ 训练用户数: 160000
   ✅ 验证用户数: 40000
   ✅ 验证集答案数: 40000
   ✅ 测试集用户数: 50000

📊 数据加载完成


In [72]:
# ============================================================================
# 📊 获取历史点击和最后一次点击（⚠️ 关键的数据泄漏防护点！）
# ============================================================================
# 
# 离线模式下的数据流：
#   click_trn (训练用户)
#       ├── click_trn_hist: 历史点击（不含最后一次），用于构造特征
#       └── click_trn_last: 最后一次点击（标签），只用于打 Label
#   
#   click_val (验证用户)
#       ├── click_val_hist: = click_val（已经是历史，不含最后一次）
#       └── click_val_last: = val_ans（Ground Truth，只用于评估！）
#
# ⚠️ 重要：
#   - click_trn_last 和 val_ans 只能用于打 Label，不能用于构造统计特征！
#   - 构造统计特征时，只能使用 click_trn_hist 和 click_val_hist
# ============================================================================

# 训练用户：划分历史和最后一次点击
click_trn_hist, click_trn_last = get_hist_and_last_click(click_trn)
print(f"📊 训练用户历史点击数: {len(click_trn_hist)}, 最后一次点击数: {len(click_trn_last)}")

# 验证用户：click_val 本身就是历史（不含最后一次），val_ans 是标签
if click_val is not None:
    click_val_hist = click_val  # 已经是历史点击
    click_val_last = val_ans    # 用于打标签和评估
    print(f"📊 验证用户历史点击数: {len(click_val_hist)}, 验证集答案数: {len(click_val_last)}")
else:
    click_val_hist, click_val_last = None, None
    
# 测试集用户：全部作为历史（需要预测最后一次）
click_tst_hist = click_tst
print(f"📊 测试集用户点击数: {len(click_tst_hist)}")

📊 训练用户历史点击数: 729066, 最后一次点击数: 160000
📊 验证用户历史点击数: 183557, 验证集答案数: 40000
📊 测试集用户点击数: 518010


## 对训练数据做负采样

通过召回我们将数据转换成三元组的形式（user1, item1, label）的形式，观察发现正负样本差距极度不平衡，我们可以先对负样本进行下采样，下采样的目的一方面缓解了正负样本比例的问题，另一方面也减小了我们做排序特征的压力，我们在做负采样的时候又有哪些东西是需要注意的呢？

1. 只对负样本进行下采样(如果有比较好的正样本扩充的方法其实也是可以考虑的)
2. 负采样之后，保证所有的用户和文章仍然出现在采样之后的数据中
3. 下采样的比例可以根据实际情况人为的控制
4. 做完负采样之后，更新此时新的用户召回文章列表，因为后续做特征的时候可能用到相对位置的信息。

其实负采样也可以留在后面做完特征在进行，这里由于做排序特征太慢了，所以把负采样的环节提到前面了。

In [73]:
# 将召回列表转换成df的形式
def recall_dict_2_df(recall_list_dict):
    df_row_list = [] # [user, item, score]
    for user, recall_list in tqdm(recall_list_dict.items(), disable=not logger.isEnabledFor(logging.DEBUG)):
        # 新增：兼容字典格式（防止上一步保存错格式）
        if isinstance(recall_list, dict):
            recall_list = recall_list.items()
            
        for item, score in recall_list:
            df_row_list.append([user, item, score])
    
    col_names = ['user_id', 'sim_item', 'score']
    recall_list_df = pd.DataFrame(df_row_list, columns=col_names)
    
    return recall_list_df


In [74]:
# 负采样函数，这里可以控制负采样时的比例, 这里给了一个默认的值
def neg_sample_recall_data(recall_items_df, sample_rate=0.001):
    #先按照label拆分吗正负样本
    pos_data = recall_items_df[recall_items_df['label'] == 1]
    neg_data = recall_items_df[recall_items_df['label'] == 0]
    
    #打印正负样本比例
    print('pos_data_num:', len(pos_data), 'neg_data_num:', len(neg_data), 'pos/neg:', len(pos_data)/len(neg_data))
    
    # 分组采样函数
    def neg_sample_func(group_df):
        neg_num = len(group_df)
        sample_num = max(int(neg_num * sample_rate), 1) # 保证最少有一个
        sample_num = min(sample_num, 5) # 保证最多不超过5个，这里可以根据实际情况进行选择
        return group_df.sample(n=sample_num, replace=True)
    
    # 对用户进行负采样，保证所有用户都在采样后的数据中
    neg_data_user_sample = neg_data.groupby('user_id', group_keys=False).apply(neg_sample_func)
    # 对文章进行负采样，保证所有文章都在采样后的数据中
    neg_data_item_sample = neg_data.groupby('sim_item', group_keys=False).apply(neg_sample_func)
    
    # 将上述两种情况下的采样数据合并    
    neg_data_new = pd.concat([neg_data_user_sample, neg_data_item_sample]).reset_index(drop=True)
    # 由于上述两个操作是分开的，可能将两个相同的数据给重复选择了，所以需要对合并后的数据进行去重
    neg_data_new = neg_data_new.sort_values(['user_id', 'score']).drop_duplicates(['user_id', 'sim_item'], keep='last')
    
    # 将正样本数据合并
    data_new = pd.concat([pos_data, neg_data_new], ignore_index=True)
    
    return data_new


In [75]:
# 召回数据打标签
def get_rank_label_df(recall_list_df, label_df, is_test=False):
    # 测试集是没有标签了，为了后面代码同一一些，这里直接给一个负数替代
    if is_test:
        recall_list_df['label'] = -1
        return recall_list_df
    
    label_df = label_df.rename(columns={'click_article_id': 'sim_item'})#统一名称

    recall_list_df_ = recall_list_df.merge(label_df[['user_id', 'sim_item', 'click_timestamp']], \
                                               how='left', on=['user_id', 'sim_item'])
#有 timestamp → 说明 (user, item) 在 label 表中存在，说明点击过
    recall_list_df_['label'] = recall_list_df_['click_timestamp'].apply(lambda x: 0.0 if np.isnan(x) else 1.0)
                                                         
    del recall_list_df_['click_timestamp']
    
    return recall_list_df_


##从召回列表中筛选训练用户，验证用户和测试用户，并且根据最后一次点击打标签并且进行负采样的过程。

In [76]:
# ============================================================================
# 📊 从召回列表中筛选用户并打标签（⚠️ 重要的数据流控制点！）
# ============================================================================
# 
# 数据流说明：
#   离线模式：
#     - 训练用户：用 click_trn_last（最后一次点击）打标签
#     - 验证用户：用 click_val_last = val_ans 打标签
#     - 测试用户：标签为 -1（实际提交时不需要标签）
#   
#   在线模式：
#     - 训练用户：用 click_trn_last 打标签
#     - 测试用户：标签为 -1
# ============================================================================

def get_user_recall_item_label_df(click_trn_hist, click_val_hist, click_tst_hist,
                                   click_trn_last, click_val_last, recall_list_df,
                                   mode="offline"):
    """
    从召回列表中筛选训练用户、验证用户和测试用户，并打标签和负采样
    
    参数:
        click_trn_hist: 训练用户的历史点击（不含最后一次）
        click_val_hist: 验证用户的历史点击（不含最后一次）
        click_tst_hist: 测试用户的点击（全部，因为要预测下一次）
        click_trn_last: 训练用户的最后一次点击（用于打 Label）
        click_val_last: 验证用户的最后一次点击（用于打 Label）
        recall_list_df: 召回列表 DataFrame
        mode: 运行模式
    
    返回:
        trn_user_item_label_df: 训练数据（含负采样）
        val_user_item_label_df: 验证数据（不做负采样，用于完整评估）
        tst_user_item_label_df: 测试数据（标签为 -1）
    """
    print("\n" + "=" * 70)
    print("📊 处理召回数据：筛选用户、打标签、负采样")
    print("=" * 70)
    
    # ─────────────────────────────────────────────────────────────────────
    # 1. 训练用户：筛选 + 打标签 + 负采样
    # ─────────────────────────────────────────────────────────────────────
    trn_users = click_trn_hist['user_id'].unique()
    trn_user_items_df = recall_list_df[recall_list_df['user_id'].isin(trn_users)]
    print(f"📊 训练用户召回候选数: {len(trn_user_items_df)}")
    
    # 打标签（用 click_trn_last）
    trn_user_item_label_df = get_rank_label_df(trn_user_items_df, click_trn_last, is_test=False)
    
    # 负采样
    trn_user_item_label_df = neg_sample_recall_data(trn_user_item_label_df)
    print(f"📊 训练用户负采样后数据量: {len(trn_user_item_label_df)}")
    
    # ─────────────────────────────────────────────────────────────────────
    # 2. 验证用户（仅离线模式）：筛选 + 打标签（不做负采样，保证完整评估）
    # ─────────────────────────────────────────────────────────────────────
    if mode == "offline" and click_val_hist is not None:
        val_users = click_val_hist['user_id'].unique()
        val_user_items_df = recall_list_df[recall_list_df['user_id'].isin(val_users)]
        print(f"📊 验证用户召回候选数: {len(val_user_items_df)}")
        
        # 打标签（用 click_val_last = val_ans）
        val_user_item_label_df = get_rank_label_df(val_user_items_df, click_val_last, is_test=False)
        # ⚠️ 验证集不做负采样，保证评估时能看到全部候选
        print(f"📊 验证用户数据量（不负采样）: {len(val_user_item_label_df)}")
    else:
        val_user_item_label_df = None
        
    # ─────────────────────────────────────────────────────────────────────
    # 3. 测试用户：筛选 + 打 -1 标签（无真实标签）
    # ─────────────────────────────────────────────────────────────────────
    tst_users = click_tst_hist['user_id'].unique()
    tst_user_items_df = recall_list_df[recall_list_df['user_id'].isin(tst_users)]
    print(f"📊 测试用户召回候选数: {len(tst_user_items_df)}")
    
    tst_user_item_label_df = get_rank_label_df(tst_user_items_df, None, is_test=True)
    
    print("=" * 70)
    
    return trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df

In [77]:
# 读取召回列表
recall_list_dict = get_recall_list(save_path, single_recall_model=None, multi_recall=True) # 
# 将召回数据转换成df
recall_list_df = recall_dict_2_df(recall_list_dict)


📂 读取多路召回列表: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results/final_recall_items_dict.pkl


In [78]:
# ============================================================================
# 📊 执行打标签和负采样
# ============================================================================

trn_user_item_label_df, val_user_item_label_df, tst_user_item_label_df = get_user_recall_item_label_df(
    click_trn_hist, 
    click_val_hist, 
    click_tst_hist,
    click_trn_last, 
    click_val_last, 
    recall_list_df,
    mode=MODE
)


📊 处理召回数据：筛选用户、打标签、负采样
📊 训练用户召回候选数: 7420868
pos_data_num: 14281 neg_data_num: 7406587 pos/neg: 0.001928148552092887
📊 训练用户负采样后数据量: 271777
📊 验证用户召回候选数: 5585672
📊 验证用户数据量（不负采样）: 5585672
📊 测试用户召回候选数: 0


In [79]:
trn_user_item_label_df.label


0         1.0
1         1.0
2         1.0
3         1.0
4         1.0
         ... 
271772    0.0
271773    0.0
271774    0.0
271775    0.0
271776    0.0
Name: label, Length: 271777, dtype: float64

## 将召回数据转换成字典

In [80]:
# 将最终的召回的df数据转换成字典的形式做排序特征
def make_tuple_func(group_df):
    row_data = []
    for name, row_df in group_df.iterrows():
        row_data.append((row_df['sim_item'], row_df['score'], row_df['label']))
    
    return row_data


In [81]:
trn_user_item_label_tuples_dict = trn_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()

if val_user_item_label_df is not None:
    val_user_item_label_tuples_dict = val_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()
else:
    val_user_item_label_tuples_dict = None
    
tst_user_item_label_tuples_dict = tst_user_item_label_df.groupby('user_id').apply(make_tuple_func).to_dict()


## 用户历史行为相关特征

对于每个用户召回的每个商品， 做特征。 具体步骤如下：
* 对于每个用户， 获取最后点击的N个商品的item_id， 
    * 对于该用户的每个召回商品， 计算与上面最后N次点击商品的相似度的和(最大， 最小，均值)， 时间差特征，相似性特征，字数差特征，与该用户的相似性特征

In [82]:
# 下面基于data做历史相关的特征
def create_feature(users_id, recall_list, click_hist_df,  articles_info, articles_emb, user_emb=None, N=1):
    """
    基于用户的历史行为做相关特征
    :param users_id: 用户id
    :param recall_list: 对于每个用户召回的候选文章列表
    :param click_hist_df: 用户的历史点击信息
    :param articles_info: 文章信息
    :param articles_emb: 文章的embedding向量, 这个可以用item_content_emb, item_w2v_emb, item_youtube_emb
    :param user_emb: 用户的embedding向量， 这个是user_youtube_emb, 如果没有也可以不用， 但要注意如果要用的话， articles_emb就要用item_youtube_emb的形式， 这样维度才一样
    :param N: 最近的N次点击  由于testA日志里面很多用户只存在一次历史点击， 所以为了不产生空值，默认是1
    """
    
    # 建立一个二维列表保存结果， 后面要转成DataFrame
    all_user_feas = []
    i = 0
    for user_id in tqdm(users_id, disable=not logger.isEnabledFor(logging.DEBUG)):
        # 该用户的最后N次点击
        hist_user_items = click_hist_df[click_hist_df['user_id']==user_id]['click_article_id'][-N:]
        
        # 遍历该用户的召回列表
        for rank, (article_id, score, label) in enumerate(recall_list[user_id]):
            # 该文章建立时间, 字数
            a_create_time = articles_info[articles_info['article_id']==article_id]['created_at_ts'].values[0]
            a_words_count = articles_info[articles_info['article_id']==article_id]['words_count'].values[0]
            single_user_fea = [user_id, article_id]
            # 计算与最后点击的商品的相似度的和， 最大值和最小值， 均值
            sim_fea = []
            time_fea = []
            word_fea = []
            # 遍历用户的最后N次点击文章
            for hist_item in hist_user_items:
                b_create_time = articles_info[articles_info['article_id']==hist_item]['created_at_ts'].values[0]
                b_words_count = articles_info[articles_info['article_id']==hist_item]['words_count'].values[0]
                
                sim_fea.append(np.dot(articles_emb[hist_item], articles_emb[article_id]))
                time_fea.append(abs(a_create_time-b_create_time))
                word_fea.append(abs(a_words_count-b_words_count))
                
            single_user_fea.extend(sim_fea)      # 相似性特征
            single_user_fea.extend(time_fea)    # 时间差特征
            single_user_fea.extend(word_fea)    # 字数差特征
            single_user_fea.extend([max(sim_fea), min(sim_fea), sum(sim_fea), sum(sim_fea) / len(sim_fea)])  # 相似性的统计特征
            
            if user_emb:  # 如果用户向量有的话， 这里计算该召回文章与用户的相似性特征 
                single_user_fea.append(np.dot(user_emb[user_id], articles_emb[article_id]))
                
            single_user_fea.extend([score, rank, label])    
            # 加入到总的表中
            all_user_feas.append(single_user_fea)
    
    # 定义列名
    id_cols = ['user_id', 'click_article_id']
    sim_cols = ['sim' + str(i) for i in range(N)]
    time_cols = ['time_diff' + str(i) for i in range(N)]
    word_cols = ['word_diff' + str(i) for i in range(N)]
    sat_cols = ['sim_max', 'sim_min', 'sim_sum', 'sim_mean']
    user_item_sim_cols = ['user_item_sim'] if user_emb else []
    user_score_rank_label = ['score', 'rank', 'label']
    cols = id_cols + sim_cols + time_cols + word_cols + sat_cols + user_item_sim_cols + user_score_rank_label
            
    # 转成DataFrame
    df = pd.DataFrame( all_user_feas, columns=cols)
    
    return df


In [83]:
# ============================================================================
# 📊 加载 Embedding（⚠️ 确保 Embedding 训练也不包含未来信息）
# ============================================================================
# 
# ⚠️ Word2Vec Embedding 训练说明：
#   - 离线模式：应使用 click_trn_hist + click_val_hist 训练
#   - 在线模式：应使用 click_trn_hist + click_tst 训练
#   这样可以确保 Embedding 中不包含"标签"信息
# ============================================================================

article_info_df = get_article_info_df()

# 用于训练 Word2Vec 的点击数据（只包含历史，不包含标签）
if MODE == "offline":
    all_click_for_emb = pd.concat([click_trn_hist, click_val_hist]).reset_index(drop=True)
else:
    all_click_for_emb = click_trn_hist.copy()

item_content_emb_dict, item_w2v_emb_dict, item_youtube_emb_dict, user_youtube_emb_dict = get_embedding(
    save_path, all_click_for_emb
)

print(f"📊 Embedding 加载完成")

-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min
📊 Embedding 加载完成


In [84]:
# ============================================================================
# 📊 构造用户-文章交互特征（⚠️ 使用正确的历史数据！）
# ============================================================================
# 
# 特征构造时使用的历史数据：
#   - 训练用户：click_trn_hist（不含最后一次点击）
#   - 验证用户：click_val_hist = click_val（不含最后一次点击）
#   - 测试用户：click_tst_hist = click_tst（全部历史）
# ============================================================================

print("\n" + "=" * 70)
print("📊 构造用户-文章交互特征（基于历史点击）")
print("=" * 70)

# 训练用户特征
print("📊 处理训练用户特征...")
trn_user_item_feats_df = create_feature(
    trn_user_item_label_tuples_dict.keys(), 
    trn_user_item_label_tuples_dict, 
    click_trn_hist,  # ⚠️ 使用历史点击，不含最后一次
    article_info_df, 
    item_content_emb_dict
)

# 验证用户特征（仅离线模式）
if val_user_item_label_tuples_dict is not None:
    print("📊 处理验证用户特征...")
    val_user_item_feats_df = create_feature(
        val_user_item_label_tuples_dict.keys(), 
        val_user_item_label_tuples_dict, 
        click_val_hist,  # ⚠️ 使用历史点击，不含最后一次
        article_info_df, 
        item_content_emb_dict
    )
else:
    val_user_item_feats_df = None

# 测试用户特征
print("📊 处理测试用户特征...")
tst_user_item_feats_df = create_feature(
    tst_user_item_label_tuples_dict.keys(), 
    tst_user_item_label_tuples_dict, 
    click_tst_hist,  # 测试集全部作为历史
    article_info_df, 
    item_content_emb_dict
)

print("=" * 70)
print("📊 用户-文章交互特征构造完成")
print("=" * 70)


📊 构造用户-文章交互特征（基于历史点击）
📊 处理训练用户特征...
📊 处理验证用户特征...
📊 处理测试用户特征...
📊 用户-文章交互特征构造完成


In [85]:
# ============================================================================
# 📊 保存用户-文章交互特征
# ============================================================================

print("📂 保存用户-文章交互特征...")

trn_user_item_feats_df.to_csv(save_path / 'trn_user_item_feats_df.csv', index=False)
print(f"   ✅ 训练特征: {save_path / 'trn_user_item_feats_df.csv'}")

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path / 'val_user_item_feats_df.csv', index=False)
    print(f"   ✅ 验证特征: {save_path / 'val_user_item_feats_df.csv'}")

tst_user_item_feats_df.to_csv(save_path / 'tst_user_item_feats_df.csv', index=False)
print(f"   ✅ 测试特征: {save_path / 'tst_user_item_feats_df.csv'}")

📂 保存用户-文章交互特征...
   ✅ 训练特征: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results/trn_user_item_feats_df.csv
   ✅ 验证特征: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results/val_user_item_feats_df.csv
   ✅ 测试特征: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results/tst_user_item_feats_df.csv


## 用户和文章特征

### 用户相关特征

这一块，正式进行特征工程，既要拼接上已有的特征， 也会做更多的特征出来，我们来梳理一下已有的特征和可构造特征：
1. 文章自身的特征， 文章字数，文章创建时间， 文章的embedding （articles表中)
2. 用户点击环境特征， 那些设备的特征(这个在df中)
3. 对于用户和商品还可以构造的特征：
    * 基于用户的点击文章次数和点击时间构造可以表现用户活跃度的特征
    * 基于文章被点击次数和时间构造可以反映文章热度的特征
    * 用户的时间统计特征： 根据其点击的历史文章列表的点击时间和文章的创建时间做统计特征，比如求均值， 这个可以反映用户对于文章时效的偏好
    * 用户的主题爱好特征， 对于用户点击的历史文章主题进行一个统计， 然后对于当前文章看看是否属于用户已经点击过的主题
    * 用户的字数爱好特征， 对于用户点击的历史文章的字数统计， 求一个均值

In [86]:
click_tst.head()


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,249999,160974,1506959142820,4,1,17,1,13,2
1,249999,160417,1506959172820,4,1,17,1,13,2
2,249998,160974,1506959056066,4,1,12,1,13,2
3,249998,202557,1506959086066,4,1,12,1,13,2
4,249997,183665,1506959088613,4,1,17,1,15,5


In [87]:
# ============================================================================
# 📊 此单元格已废弃，统计特征构造逻辑已移至下一个单元格
# ============================================================================
# 原因：需要严格控制用于统计的数据，防止数据泄漏
# 新逻辑见下一个单元格
# ============================================================================

In [88]:
# ============================================================================
# 📊 构造用户/文章统计特征（⚠️ 最关键的数据泄漏防护点！）
# ============================================================================
# 
# ⚠️ 数据泄漏风险分析：
#   构造用户活跃度、文章热度等统计特征时，如果使用了"标签"数据（最后一次点击），
#   就会造成数据泄漏，导致线下效果虚高，线上效果差。
#
# ✅ 正确做法：
#   1. 离线模式：只使用 click_trn_hist + click_val_hist（不含任何最后一次点击）
#   2. 在线模式：训练用户用 click_trn_hist，测试用户用 click_tst（全部历史）
#
# ============================================================================

articles = pd.read_csv(data_path / 'articles.csv')
articles = reduce_mem(articles)

# ═══════════════════════════════════════════════════════════════════════════
# 🔐 根据模式构造"纯历史"数据（不含任何标签信息）
# ═══════════════════════════════════════════════════════════════════════════

if MODE == "offline":
    # ─────────────────────────────────────────────────────────────────────
    # 🔬 离线模式：使用训练用户历史 + 验证用户历史
    #    ⚠️ 注意：click_trn_hist 是训练用户去掉最后一次点击后的历史
    #            click_val_hist 是验证用户去掉最后一次点击后的历史
    #            这样可以确保统计特征不包含任何"未来"信息
    # ─────────────────────────────────────────────────────────────────────
    print("📊 [离线模式] 构造统计特征，使用数据：")
    print("   - 训练用户历史点击（click_trn_hist）：不含训练用户的最后一次点击")
    print("   - 验证用户历史点击（click_val_hist）：不含验证用户的最后一次点击")
    print("   ⚠️ 确保无未来信息泄漏！")
    
    # 合并用于统计的数据（只包含历史，不包含标签）
    all_data_for_stats = pd.concat([click_trn_hist, click_val_hist]).reset_index(drop=True)
    # 测试集的数据也可以加入统计（因为测试集本身就是历史）
    all_data_for_stats = pd.concat([all_data_for_stats, click_tst]).reset_index(drop=True)
    
else:
    # ─────────────────────────────────────────────────────────────────────
    # 🚀 在线模式：使用训练集历史 + 测试集全部
    # ─────────────────────────────────────────────────────────────────────
    print("📊 [在线模式] 构造统计特征，使用数据：")
    print("   - 训练用户历史点击（click_trn_hist）：不含训练用户的最后一次点击")
    print("   - 测试集全部点击（click_tst）")
    
    all_data_for_stats = pd.concat([click_trn_hist, click_tst]).reset_index(drop=True)

# 拼上文章信息
all_data = all_data_for_stats.merge(articles, left_on='click_article_id', right_on='article_id')

print(f"\n📊 用于统计特征的数据量: {len(all_data)}")
print(f"📊 涉及用户数: {all_data['user_id'].nunique()}")
print(f"📊 涉及文章数: {all_data['click_article_id'].nunique()}")

-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min
📊 [离线模式] 构造统计特征，使用数据：
   - 训练用户历史点击（click_trn_hist）：不含训练用户的最后一次点击
   - 验证用户历史点击（click_val_hist）：不含验证用户的最后一次点击
   ⚠️ 确保无未来信息泄漏！

📊 用于统计特征的数据量: 1430633
📊 涉及用户数: 250000
📊 涉及文章数: 31014


In [89]:
all_data.shape


(1430633, 13)

#### 分析一下点击时间和点击文章的次数，区分用户活跃度
如果某个用户点击文章之间的时间间隔比较小， 同时点击的文章次数很多的话， 那么我们认为这种用户一般就是活跃用户, 当然衡量用户活跃度的方式可能多种多样， 这里我们只提供其中一种，我们写一个函数， 得到可以衡量用户活跃度的特征，逻辑如下：
1. 首先根据用户user_id分组， 对于每个用户，计算点击文章的次数， 两两点击文章时间间隔的均值
2. 把点击次数取倒数和时间间隔的均值统一归一化，然后两者相加合并，该值越小， 说明用户越活跃
3. 注意， 上面两两点击文章的时间间隔均值， 会出现如果用户只点击了一次的情况，这时候时间间隔均值那里会出现空值， 对于这种情况最后特征那里给个大数进行区分

这个的衡量标准就是先把点击的次数取到数然后归一化， 然后点击的时间差归一化， 然后两者相加进行合并， 该值越小， 说明被点击的次数越多， 且间隔时间短。

In [90]:
# ============================================================================
# 📊 用户活跃度特征
# ============================================================================
# ⚠️ 使用的数据：all_data（只包含历史点击，不含标签）
#    确保无数据泄漏
# ============================================================================

def active_level(all_data, cols):
    """
    制作衡量用户活跃度的特征
    
    逻辑：
    1. 点击次数越多，活跃度越高
    2. 点击时间间隔越短，活跃度越高
    
    参数:
        all_data: 用于统计的数据（⚠️ 必须只包含历史点击，不含标签）
        cols: 用到的特征列
    """
    data = all_data[cols].copy()
    data.sort_values(['user_id', 'click_timestamp'], inplace=True)
    user_act = pd.DataFrame(data.groupby('user_id', as_index=False)[['click_article_id', 'click_timestamp']].\
                            agg({'click_article_id':np.size, 'click_timestamp': {list}}).values, columns=['user_id', 'click_size', 'click_timestamp'])
    
    # 计算时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([j-i for i, j in list(zip(l[:-1], l[1:]))])
        
    user_act['time_diff_mean'] = user_act['click_timestamp'].apply(lambda x: time_diff_mean(x))
    
    # 点击次数取倒数
    user_act['click_size'] = 1 / user_act['click_size']
    
    # 两者归一化
    user_act['click_size'] = (user_act['click_size'] - user_act['click_size'].min()) / (user_act['click_size'].max() - user_act['click_size'].min())
    user_act['time_diff_mean'] = (user_act['time_diff_mean'] - user_act['time_diff_mean'].min()) / (user_act['time_diff_mean'].max() - user_act['time_diff_mean'].min())     
    user_act['active_level'] = user_act['click_size'] + user_act['time_diff_mean']
    
    user_act['user_id'] = user_act['user_id'].astype('int')
    del user_act['click_timestamp']
    return user_act

In [91]:
user_act_fea = active_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])


In [92]:
user_act_fea.head()


,user_id,click_size,time_diff_mean,active_level
0,0,1.0,1.613846e-09,1.0
1,1,1.0,1.613846e-09,1.0
2,2,1.0,1.613846e-09,1.0
3,3,1.0,1.613846e-09,1.0
4,4,1.0,1.613846e-09,1.0


#### 分析一下点击时间和被点击文章的次数， 衡量文章热度特征
和上面同样的思路， 如果一篇文章在很短的时间间隔之内被点击了很多次， 说明文章比较热门，实现的逻辑和上面的基本一致， 只不过这里是按照点击的文章进行分组：
1. 根据文章进行分组， 对于每篇文章的用户， 计算点击的时间间隔
2. 将用户的数量取倒数， 然后用户的数量和时间间隔归一化， 然后相加得到热度特征， 该值越小， 说明被点击的次数越大且时间间隔越短， 文章比较热

当然， 这只是给出一种判断文章热度的一种方法， 这里大家也可以头脑风暴一下

In [93]:
# ============================================================================
# 📊 文章热度特征
# ============================================================================
# ⚠️ 使用的数据：all_data（只包含历史点击，不含标签）
#    确保无数据泄漏
# ============================================================================

def hot_level(all_data, cols):
    """
    制作衡量文章热度的特征
    
    逻辑：
    1. 被点击次数越多，热度越高
    2. 点击时间间隔越短，热度越高
    
    参数:
        all_data: 用于统计的数据（⚠️ 必须只包含历史点击，不含标签）
        cols: 用到的特征列
    """
    data = all_data[cols].copy()
    data.sort_values(['click_article_id', 'click_timestamp'], inplace=True)
    article_hot = pd.DataFrame(data.groupby('click_article_id', as_index=False)[['user_id', 'click_timestamp']].\
                            agg({'user_id':np.size, 'click_timestamp': {list}}).values, columns=['click_article_id', 'user_num', 'click_timestamp'])
    
    # 计算被点击时间间隔的均值
    def time_diff_mean(l):
        if len(l) == 1:
            return 1
        else:
            return np.mean([j-i for i, j in list(zip(l[:-1], l[1:]))])
        
    article_hot['time_diff_mean'] = article_hot['click_timestamp'].apply(lambda x: time_diff_mean(x))
    
    # 点击次数取倒数
    article_hot['user_num'] = 1 / article_hot['user_num']
    
    # 两者归一化
    article_hot['user_num'] = (article_hot['user_num'] - article_hot['user_num'].min()) / (article_hot['user_num'].max() - article_hot['user_num'].min())
    article_hot['time_diff_mean'] = (article_hot['time_diff_mean'] - article_hot['time_diff_mean'].min()) / (article_hot['time_diff_mean'].max() - article_hot['time_diff_mean'].min())     
    article_hot['hot_level'] = article_hot['user_num'] + article_hot['time_diff_mean']
    
    article_hot['click_article_id'] = article_hot['click_article_id'].astype('int')
    
    del article_hot['click_timestamp']
    
    return article_hot

In [94]:
article_hot_fea = hot_level(all_data, ['user_id', 'click_article_id', 'click_timestamp'])    


In [95]:
article_hot_fea.head()


,click_article_id,user_num,time_diff_mean,hot_level
0,69,1.0,6.627644e-13,1.0
1,84,1.0,6.627644e-13,1.0
2,94,1.0,6.627644e-13,1.0
3,137,1.0,6.627644e-13,1.0
4,142,1.0,6.627644e-13,1.0


#### 用户的系列习惯
这个基于原来的日志表做一个类似于article的那种DataFrame， 存放用户特有的信息, 主要包括点击习惯， 爱好特征之类的
* 用户的设备习惯， 这里取最常用的设备（众数）
* 用户的时间习惯： 根据其点击过得历史文章的时间来做一个统计（这个感觉最好是把时间戳里的时间特征的h特征提出来，看看用户习惯一天的啥时候点击文章）， 但这里先用转换的时间吧， 求个均值
* 用户的爱好特征， 对于用户点击的历史文章主题进行用户的爱好判别， 更偏向于哪几个主题， 这个最好是multi-hot进行编码， 先试试行不
* 用户文章的字数差特征， 用户的爱好文章的字数习惯

这些就是对用户进行分组， 然后统计即可

#### 用户的设备习惯

In [96]:
def device_fea(all_data, cols):
    """
    制作用户的设备特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_device_info = all_data[cols]
    
    # 用众数来表示每个用户的设备信息
    user_device_info = user_device_info.groupby('user_id').agg(lambda x: x.value_counts().index[0]).reset_index()
    
    return user_device_info


In [97]:
# 设备特征(这里时间会比较长)
device_cols = ['user_id', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']
user_device_info = device_fea(all_data, device_cols)


In [98]:
user_device_info.head()


,user_id,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,4,1,17,1,25,2
1,1,4,1,17,1,25,6
2,2,4,3,20,1,25,2
3,3,4,3,2,1,25,2
4,4,4,1,12,1,16,1


#### 用户的时间习惯

In [99]:
def user_time_hob_fea(all_data, cols):
    """
    制作用户的时间习惯特征
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_time_hob_info = all_data[cols]
    
    # 先把时间戳进行归一化
    mm = MinMaxScaler()
    user_time_hob_info['click_timestamp'] = mm.fit_transform(user_time_hob_info[['click_timestamp']])
    user_time_hob_info['created_at_ts'] = mm.fit_transform(user_time_hob_info[['created_at_ts']])

    user_time_hob_info = user_time_hob_info.groupby('user_id').agg('mean').reset_index()
    
    user_time_hob_info.rename(columns={'click_timestamp': 'user_time_hob1', 'created_at_ts': 'user_time_hob2'}, inplace=True)
    return user_time_hob_info


In [100]:
user_time_hob_cols = ['user_id', 'click_timestamp', 'created_at_ts']
user_time_hob_info = user_time_hob_fea(all_data, user_time_hob_cols)


#### 用户的主题爱好
这里先把用户点击的文章属于的主题转成一个列表， 后面再总的汇总的时候单独制作一个特征， 就是文章的主题如果属于这里面， 就是1， 否则就是0。

In [101]:
def user_cat_hob_fea(all_data, cols):
    """
    用户的主题爱好
    :param all_data: 数据集
    :param cols: 用到的特征列
    """
    user_category_hob_info = all_data[cols]
    user_category_hob_info = user_category_hob_info.groupby('user_id').agg({list}).reset_index()
    
    user_cat_hob_info = pd.DataFrame()
    user_cat_hob_info['user_id'] = user_category_hob_info['user_id']
    user_cat_hob_info['cate_list'] = user_category_hob_info['category_id']
    
    return user_cat_hob_info


In [102]:
user_category_hob_cols = ['user_id', 'category_id']
user_cat_hob_info = user_cat_hob_fea(all_data, user_category_hob_cols)


#### 用户的字数偏好特征

In [103]:
user_wcou_info = all_data.groupby('user_id')['words_count'].agg('mean').reset_index()
user_wcou_info.rename(columns={'words_count': 'words_hbo'}, inplace=True)


#### 用户的信息特征合并保存

In [104]:
# 所有表进行合并

user_info = pd.merge(user_act_fea, user_device_info, on='user_id')
user_info = user_info.merge(user_time_hob_info, on='user_id')
user_info = user_info.merge(user_cat_hob_info, on='user_id')
user_info = user_info.merge(user_wcou_info, on='user_id')



In [105]:
# ============================================================================
# 📂 保存用户特征
# ============================================================================
# ⚠️ 这些特征是基于"历史点击"数据计算的，不包含标签信息
# ============================================================================

user_info.to_csv(save_path / 'user_info.csv', index=False)
print(f"📂 用户特征已保存: {save_path / 'user_info.csv'}")
print(f"   用户数: {len(user_info)}")

📂 用户特征已保存: /Users/lixiang/Desktop/fun-rec-master/fun-rec-master/tmp/projects/news_recommendation/temp_results/user_info.csv
   用户数: 250000


### 用户特征直接读入
如果前面关于用户的特征工程已经给做完了，后面可以直接读取

In [106]:
# 把用户信息直接读入进来
user_info = pd.read_csv(save_path / 'user_info.csv')


In [107]:
if os.path.exists(save_path / 'trn_user_item_feats_df.csv'):
    trn_user_item_feats_df = pd.read_csv(save_path / 'trn_user_item_feats_df.csv')
    
if os.path.exists(save_path / 'tst_user_item_feats_df.csv'):
    tst_user_item_feats_df = pd.read_csv(save_path / 'tst_user_item_feats_df.csv')

if os.path.exists(save_path / 'val_user_item_feats_df.csv'):
    val_user_item_feats_df = pd.read_csv(save_path / 'val_user_item_feats_df.csv')
else:
    val_user_item_feats_df = None


In [108]:
# 拼上用户特征
# 下面是线下验证的
trn_user_item_feats_df = trn_user_item_feats_df.merge(user_info, on='user_id', how='left')

if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(user_info, on='user_id', how='left')
else:
    val_user_item_feats_df = None
    
tst_user_item_feats_df = tst_user_item_feats_df.merge(user_info, on='user_id',how='left')


In [109]:
trn_user_item_feats_df.columns


Index(['user_id', 'click_article_id', 'sim0', 'time_diff0', 'word_diff0',
       'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'label',
       'click_size', 'time_diff_mean', 'active_level', 'click_environment',
       'click_deviceGroup', 'click_os', 'click_country', 'click_region',
       'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'cate_list',
       'words_hbo'],
      dtype='object')

### 文章的特征直接读入

In [110]:
articles =  pd.read_csv(data_path / 'articles.csv')
articles = reduce_mem(articles)


-- Mem. usage decreased to  5.56 Mb (50.0% reduction),time spend:0.00 min


In [111]:
# 拼上文章特征
trn_user_item_feats_df = trn_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')

if val_user_item_feats_df is not None:
    val_user_item_feats_df = val_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')
else:
    val_user_item_feats_df = None

tst_user_item_feats_df = tst_user_item_feats_df.merge(articles, left_on='click_article_id', right_on='article_id')


### 召回文章的主题是否在用户的爱好里面

In [112]:
trn_user_item_feats_df['is_cat_hab'] = trn_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
if val_user_item_feats_df is not None:
    val_user_item_feats_df['is_cat_hab'] = val_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)
else:
    val_user_item_feats_df = None
# TODO: 这里因为是sample数据原因，tst_user_item_feats_df 大小为0，当使用全量数据时，需要删除这行
if tst_user_item_feats_df.shape[0] > 0:
    tst_user_item_feats_df['is_cat_hab'] = tst_user_item_feats_df.apply(lambda x: 1 if x.category_id in set(x.cate_list) else 0, axis=1)


In [113]:
# ============================================================================
# 📊 删除不需要的列
# ============================================================================

# 删除 cate_list 列（已用于构造 is_cat_hab 特征）
del trn_user_item_feats_df['cate_list']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['cate_list']
    
if tst_user_item_feats_df.shape[0] > 0:
    del tst_user_item_feats_df['cate_list']

# 删除 article_id 列（与 click_article_id 重复）
del trn_user_item_feats_df['article_id']

if val_user_item_feats_df is not None:
    del val_user_item_feats_df['article_id']
    
if tst_user_item_feats_df.shape[0] > 0:
    del tst_user_item_feats_df['article_id']

print("✅ 已删除不需要的列 (cate_list, article_id)")

✅ 已删除不需要的列 (cate_list, article_id)


## 保存特征

In [114]:
# ============================================================================
# 📊 保存最终特征（包含用户特征和文章特征）
# ============================================================================

print("\n" + "=" * 70)
print("📂 保存最终排序特征")
print("=" * 70)

trn_user_item_feats_df.to_csv(save_path / 'trn_user_item_feats_df.csv', index=False)
print(f"   ✅ 训练特征数据: {len(trn_user_item_feats_df)} 条")

if val_user_item_feats_df is not None:
    val_user_item_feats_df.to_csv(save_path / 'val_user_item_feats_df.csv', index=False)
    print(f"   ✅ 验证特征数据: {len(val_user_item_feats_df)} 条")

tst_user_item_feats_df.to_csv(save_path / 'tst_user_item_feats_df.csv', index=False)
print(f"   ✅ 测试特征数据: {len(tst_user_item_feats_df)} 条")

# 保存当前模式信息，方便 Ranking 阶段检查一致性
mode_info = {
    'mode': MODE,
    'trn_users': len(trn_user_item_feats_df['user_id'].unique()) if 'user_id' in trn_user_item_feats_df.columns else 0,
    'val_users': len(val_user_item_feats_df['user_id'].unique()) if val_user_item_feats_df is not None else 0,
    'tst_users': len(tst_user_item_feats_df['user_id'].unique()) if 'user_id' in tst_user_item_feats_df.columns else 0,
}
pickle.dump(mode_info, open(save_path / 'feature_eng_mode_info.pkl', 'wb'))

print("\n" + "=" * 70)
print(f"🎉 特征工程完成！模式: {MODE}")
print("=" * 70)
if MODE == "offline":
    print("📌 下一步：运行 6.ranking.ipynb 进行排序模型训练和验证")
else:
    print("📌 下一步：运行 6.ranking.ipynb 进行排序模型训练和提交")
print("=" * 70)


📂 保存最终排序特征
   ✅ 训练特征数据: 271777 条
   ✅ 验证特征数据: 5585672 条
   ✅ 测试特征数据: 0 条

🎉 特征工程完成！模式: offline
📌 下一步：运行 6.ranking.ipynb 进行排序模型训练和验证


## 总结

特征工程和数据清洗转换是比赛中至关重要的一块， 因为**数据和特征决定了机器学习的上限，而算法和模型只是逼近这个上限而已**，所以特征工程的好坏往往决定着最后的结果，**特征工程**可以一步增强数据的表达能力，通过构造新特征，我们可以挖掘出数据的更多信息，使得数据的表达能力进一步放大。 在本节内容中，我们主要是先通过制作特征和标签把预测问题转成了监督学习问题，然后围绕着用户画像和文章画像进行一系列特征的制作， 此外，为了保证正负样本的数据均衡，我们还学习了负采样就技术等。当然本节内容只是对构造特征提供了一些思路，也请学习者们在学习过程中开启头脑风暴，尝试更多的构造特征的方法，也欢迎我们一块探讨和交流。